## 1. Siapkan Proyek

Mengunduh kode dari repositori, lalu menghubungkan Google Drive agar hasil
scraping tidak hilang saat runtime Colab di-reset.

Tidak perlu upload apa pun secara manual — cukup jalankan sel di bawah.


---
## ⚠️ Sebelum Membagikan Notebook Ini

Notebook ini aman dibagikan **selama tiga hal berikut dipatuhi**:

**1. Bagikan link NOTEBOOK saja, bukan folder Drive.**  
Folder `scraper-x` di Drive Anda berisi `session/cookies.json` — file itu
setara password akun X Anda. Kalau foldernya ikut dibagikan, akun Anda bocor.

**2. Bersihkan output dulu:** menu **Edit → Clear all outputs**, lalu simpan.  
Output sel ikut tersimpan di dalam file notebook.

**3. Atur izin berbagi ke *Viewer*, bukan *Editor*.**  
Dengan izin Editor, orang lain dapat menyisipkan kode yang mengirim cookie
Anda ke pihak ketiga. Minta mereka **File → Save a copy in Drive** untuk
membuat salinan sendiri.

---
### Untuk pengguna yang menerima notebook ini

Anda memakai **akun X dan Google Drive sendiri**. Cookie yang Anda upload
tersimpan di Drive Anda sendiri dan tidak terlihat oleh siapa pun.

Langkahnya: **File → Save a copy in Drive** → siapkan folder proyek di
My Drive → jalankan sel dari atas.

> Catatan: scraping otomatis melanggar Terms of Service X. Gunakan hanya
> untuk keperluan akademis, dan sadari akun yang dipakai menanggung risikonya.


## 1. Siapkan Proyek

Mengunduh kode dari repositori, lalu menghubungkan Google Drive agar hasil
scraping tidak hilang saat runtime Colab di-reset.

Tidak perlu upload apa pun secara manual — cukup jalankan sel di bawah.


In [ ]:
import os, shutil
from google.colab import drive

REPO = 'https://codeberg.org/fahmikemal/scrapper-x.git'
KERJA = '/content/drive/MyDrive/scraper-x'   # hasil disimpan di Drive Anda

drive.mount('/content/drive')

if os.path.exists(os.path.join(KERJA, 'main.py')):
    print('Proyek sudah ada di Drive, mengambil pembaruan...')
    os.chdir(KERJA)
    !git pull --ff-only 2>&1 | tail -3
else:
    print('Mengunduh proyek dari repositori...')
    !git clone --depth 1 $REPO /content/_src 2>&1 | tail -2
    os.makedirs(KERJA, exist_ok=True)
    !cp -r /content/_src/. $KERJA/
    os.chdir(KERJA)

# Folder kerja (diabaikan git, aman untuk data pribadi Anda)
for d in ['data/raw','data/processed','data/labeled','data/gold',
          'models','results','session']:
    os.makedirs(d, exist_ok=True)

print('\nDirektori kerja:', os.getcwd())
print('main.py   :', 'OK' if os.path.exists('main.py') else 'HILANG')
print('lexicon   :', 'OK' if os.path.exists('lexicon/positive.tsv') else 'HILANG')


## 2. Install Library

Cukup sekali per sesi Colab. Kalau runtime di-restart, jalankan lagi.

In [ ]:
!pip install -q -r requirements.txt
!playwright install-deps chromium 2>/dev/null | tail -2
!playwright install chromium 2>&1 | tail -2

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('\nInstalasi selesai.')

In [ ]:
# Verifikasi semua siap
import importlib.metadata as md
for p in ['playwright','pandas','scikit-learn','nltk','PySastrawi','langdetect']:
    try:    print(f'  {p:14} {md.version(p)}')
    except: print(f'  {p:14} BELUM TERPASANG')

import os
print('\nLexicon InSet :', 'OK' if os.path.exists('lexicon/positive.tsv') else 'HILANG')
print('Cookie X      :', 'OK' if os.path.exists('session/cookies.json') else 'BELUM (lihat sel 3)')

## 3. Upload Cookie Session X

**Lakukan di browser Anda sendiri, bukan di Colab:**

1. Pasang ekstensi **Cookie-Editor** di Chrome/Edge
2. Buka <https://x.com> dan login seperti biasa
3. Klik ikon Cookie-Editor → tombol **Export** (tersalin ke clipboard)
4. Buat file `cookies.json` di komputer, paste isinya, simpan
5. Jalankan sel di bawah lalu pilih file tersebut

> Cookie berisi `auth_token` — setara akses penuh ke akun X Anda.
> Jangan bagikan file ini, dan jangan bagikan notebook ini setelah cookie diupload.

In [ ]:
import os, json, shutil
from google.colab import files

os.makedirs('session', exist_ok=True)
print('Pilih file cookies.json Anda...')
up = files.upload()

nama = list(up.keys())[0]
shutil.move(nama, 'session/cookies.json')
os.chmod('session/cookies.json', 0o600)

d = json.load(open('session/cookies.json', encoding='utf-8'))
nm = {c.get('name') for c in d}
print(f'\nTersimpan: {len(d)} cookie')
for w in ('auth_token', 'ct0'):
    print(f'  {w:12}: {"ADA" if w in nm else "TIDAK ADA -> cookie tidak lengkap"}')
print('\nCatatan: file ini tersimpan di Drive ANDA sendiri.')
print('Jangan bagikan folder Drive yang memuatnya.')


## 4. Scraping

Mengambil komentar dari **33 kata kunci** layanan Traveloka.

- `--max 130` → sekitar 1.500–2.000 tweet mentah → **±800–1.000 data Indonesia**
- Target skripsi minimal **500** data
- **Progres disimpan otomatis setiap query selesai**, jadi kalau Colab
  terputus, data yang sudah terkumpul tetap aman

> Naikkan `--max` bila ingin lebih banyak. Turunkan bila ingin cepat.

In [ ]:
!python main.py --scrape --max 130 --headless

In [ ]:
# Cek hasil scraping
import pandas as pd, glob
f = sorted(glob.glob('data/raw/*.csv'))
if not f:
    print('Belum ada hasil scraping.')
else:
    for x in f:
        d = pd.read_csv(x, encoding='utf-8-sig')
        print(f'{x}: {len(d)} tweet | {d["date"].min()[:10]} s/d {d["date"].max()[:10]}')
    df = pd.read_csv(f[-1], encoding='utf-8-sig')
    display(df.head())

### Kalau data masih kurang dari 500

Jalankan lagi sel scraping di atas (boleh di hari berikutnya — konten X
bertambah terus), lalu gunakan `--merge` di tahap preprocessing untuk
menggabungkan semua sesi. Duplikat dibuang otomatis berdasarkan ID tweet.

## 5. Preprocessing

Cleaning → **filter bahasa** → tokenisasi → stopword removal → stemming.

Dokumen non-Indonesia dibuang karena Sastrawi dan InSet hanya valid untuk
Bahasa Indonesia. Kata negasi (`tidak`, `bukan`, `gak`) **sengaja dipertahankan**
agar model dapat mempelajari pembalikan polaritas.

> Tambahkan `--merge` bila Anda scraping lebih dari sekali.

In [ ]:
!python main.py --preprocess
# Bila scraping lebih dari sekali, pakai:
# !python main.py --preprocess --merge

## 6. Pelabelan Otomatis (Lexicon InSet)

InSet: 3.607 kata positif + 6.606 kata negatif, bobot −5..+5, ditambah
`lexicon/custom.tsv` untuk istilah domain yang tidak ada di InSet
(`kecewa`, `tipu`, `refund`, `komplain`).

Label: `0` negatif · `1` netral · `2` positif. Ambang netral `|skor| ≤ 2`.

In [ ]:
!python main.py --autolabel

## 7–9. Validasi Label — **JANGAN DILEWATI**

Pelabelan otomatis **bukan ground truth**, hanya perkiraan. Validitasnya
wajib diukur terhadap label manusia memakai **Cohen's Kappa**.

Inilah bagian yang membuat hasil penelitian Anda dapat dipertanggungjawabkan
di sidang.

In [ ]:
# 7. Ambil 200 sampel acak (stratified) untuk dilabeli manual
!python main.py --sample-gold 200

### 8. Label manual (dikerjakan Anda sendiri)

1. Buka Google Drive → `scraper-x/data/gold/gold_sample_*.csv`
2. Buka dengan **Google Sheets**
3. Isi kolom **`label_manual`**: `0` negatif · `1` netral · `2` positif
4. Baca kolom `text` (teks asli). **Jangan lihat kolom `label_auto`** dulu —
   supaya penilaian Anda tidak terpengaruh
5. File → Download → **Comma Separated Values (.csv)**
6. Upload kembali menimpa file aslinya di folder `data/gold/`

> Ini pekerjaan paling memakan waktu (1–3 jam), tapi tidak bisa diwakilkan ke
> program. Justru penilaian manual Anda inilah yang menjadi acuan kebenaran.

In [ ]:
# 9. Ukur kesepakatan label otomatis vs label manual
!python main.py --kappa

**Interpretasi Cohen's Kappa (Landis & Koch, 1977):**

| Kappa | Arti | Tindakan |
|---|---|---|
| < 0.20 | sangat rendah | jangan pakai label otomatis |
| 0.20 – 0.40 | rendah | perbaiki lexicon/ambang dulu |
| 0.40 – 0.60 | sedang | dapat dipakai, sebutkan sebagai keterbatasan |
| 0.60 – 0.80 | baik | aman dipakai |
| > 0.80 | sangat baik | aman dipakai |

Berapa pun hasilnya, **laporkan apa adanya di skripsi**. Kappa rendah bukan
kegagalan — itu temuan yang jujur tentang keterbatasan pelabelan berbasis lexicon.

## 10. Training Model

1. **Split dulu** (80/20, stratified) — data uji dibiarkan timpang sesuai kondisi nyata
2. **Oversampling hanya pada data latih** — mencegah data leakage
3. **Baseline kelas mayoritas** sebagai acuan minimum
4. TF-IDF (unigram + bigram) → GridSearch alpha → **MultinomialNB** (utama) dan
   **ComplementNB** (pembanding)
5. Cross-validation 10-fold dengan oversampling **di dalam tiap fold**

In [ ]:
!python main.py --train

In [ ]:
# Tampilkan confusion matrix dan tabel metrik
import pandas as pd, glob
from IPython.display import Image, display

m = sorted(glob.glob('results/metrics_*.csv'))
if m:
    display(pd.read_csv(m[-1], encoding='utf-8-sig'))

for img in sorted(glob.glob('results/confusion_matrix_*.png'))[-2:]:
    print(img)
    display(Image(img))

---

## Catatan untuk Penulisan Skripsi

**Metrik utama adalah macro-F1, bukan accuracy.** Pada data timpang, accuracy
menyesatkan — model yang selalu menebak kelas mayoritas bisa terlihat bagus
tanpa mempelajari apa pun. Karena itu setiap run mencetak baseline kelas
mayoritas; model hanya berguna bila melampauinya.

**Yang wajib dilaporkan:**

1. Jumlah data mentah, jumlah yang dibuang filter bahasa, jumlah akhir
2. Nilai **Cohen's Kappa** beserta interpretasinya
3. Bagian *"Dilewati / gagal"* dari laporan scraping — sebagai keterbatasan
   pengumpulan data
4. Isi `lexicon/custom.tsv` — sebutkan sebagai *"InSet augmented with domain terms"*
5. Alasan operator `lang:id` tidak dipakai (lihat komentar di `main.py`)
6. Perbandingan MultinomialNB vs ComplementNB

**Keterbatasan yang perlu diakui:**

- Pelabelan lexicon bukan ground truth
- `langdetect` kurang akurat pada teks pendek
- Sampel dari tab *Latest* bersifat bias-kebaruan, bukan acak
- Scraping X melanggar ToS X — gunakan hanya untuk keperluan akademis, dan
  pertimbangkan anonimisasi kolom `username` pada lampiran